# 1. 讀取檔案

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
import numpy as np

# 使用 pandas 的 read_csv 函式讀取訓練資料
df = pd.read_csv("data/train.csv")

# 2.資料預處理

In [9]:
# 直接刪除任何含有缺失值的整行資料
# 缺失值填補
imputer = SimpleImputer(strategy='median')
df[['horsepower']] = imputer.fit_transform(df[['horsepower']])
# one-hot origin
df_clean = pd.get_dummies(df, columns=["origin"])

# 3.特徵工程

In [10]:
# 定義要用來預測的特徵欄位
# 特徵工程
df_clean['power_to_weight'] = df_clean['horsepower'] / df_clean['weight']
df_clean['displacement_per_cyl'] = df_clean['displacement'] / df_clean['cylinders']

# ===== 2. 定義特徵與目標 =====
features = [
    'weight', 'acceleration', 'model_year',
    'displacement', 'horsepower', 'cylinders',
    'origin_usa', 'origin_japan', 'origin_europe',
    'power_to_weight', 'displacement_per_cyl'
]

target = 'mpg'

# 從乾淨的資料中選取 X 和 y
X = df_clean[features]
y = df_clean[target] 


# 4. 訓練模型

In [17]:
# 分割訓練集與測試集 (用於本地評估模型好壞)
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 建立並訓練模型
model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    min_child_weight=3,
    reg_lambda=2.0,
    subsample=0.8,
    colsample_bytree=0.6,
    reg_alpha=0.1,
    random_state=42
)

model.fit(X_train, y_train)

# 進行預測與評估
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)
train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print("模型已訓練完成！")
print(f"訓練誤差 (Train RMSE): {train_rmse:.4f} MPG")
print(f"測試誤差 (Test RMSE):  {test_rmse:.4f} MPG")

# 各個特徵的權重
for feature, coef in zip(features, model.feature_importances_):
    print(f"特徵 '{feature}' 的權重: {coef:.4f}")


模型已訓練完成！
訓練誤差 (Train RMSE): 1.4660 MPG
測試誤差 (Test RMSE):  2.9126 MPG
特徵 'weight' 的權重: 0.0795
特徵 'acceleration' 的權重: 0.0123
特徵 'model_year' 的權重: 0.0742
特徵 'displacement' 的權重: 0.2033
特徵 'horsepower' 的權重: 0.1160
特徵 'cylinders' 的權重: 0.4478
特徵 'origin_usa' 的權重: 0.0091
特徵 'origin_japan' 的權重: 0.0001
特徵 'origin_europe' 的權重: 0.0170
特徵 'power_to_weight' 的權重: 0.0127
特徵 'displacement_per_cyl' 的權重: 0.0281


# 5.輸出提交檔案

In [14]:
# 讀取需要進行預測的測試檔案 test.csv
df_test = pd.read_csv("data/test.csv")
df_test = pd.get_dummies(df_test, columns=["origin"])
df_test['power_to_weight'] = df_test['horsepower'] / df_test['weight']
df_test['displacement_per_cyl'] = df_test['displacement'] / df_test['cylinders']

# 對測試資料進行預處理
# HINT：如果前面使用了其他的預處理方式，這邊要如何修改？
df_test = df_test.fillna(0)

# 使用訓練好的模型，對測試資料進行預測
predictions = model.predict(df_test[features])

# 建立一個新的 DataFrame
submission_df = pd.DataFrame({'Id':df_test['id'], 'mpg': predictions})

# 保存為 submission.csv
submission_df.to_csv('submission.csv', index=False)
print("提交文件 'submission.csv' 已成功生成！")

提交文件 'submission.csv' 已成功生成！


# 6. 報告

姓名：__________ 學號：__________

第一部分：準確度分數 (Accuracy Scores) (1分)  
我的準確度分數：__________  

第二部分：我的實驗記錄 (My Experiment Log) (3分)  
請記錄你做了哪些嘗試來提升分數，至少記錄兩次不同的嘗試。  
【實驗 1】  
    我做的修改：__________________________________________________________________________________  
    結果與觀察 (分數變化、心得等)：__________________________________________________________________  
    該次實驗分數： ____________  
【實驗 2】  
    我做的修改：__________________________________________________________________________________  
    結果與觀察 (分數變化、心得等)：__________________________________________________________________  
    該次實驗分數： ____________  

第三部分：總結與心得 (Conclusion & Reflection) (2分)  
請撰寫一段約 50-100 字的心得總結。內容需包含：  
(1) 你認為本次實驗中，提升準確率最有效的修改是什麼。  
(2) 這次不斷嘗試與修正的過程，帶給你最大的學習與啟發。  
內容：______________________________________________    